In [ ]:
print('STEP 2: Interval Parsing (CHUNKS + STREAMING)')
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

In [50]:
base_path = Path(r'C:/Users/rroman/OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)/Documents/Unidad/2026/Capacitación/Data Science y Machine Learning/proyecto final/data')
input_path = base_path / 'interim' / 'df_final.parquet'
output_path = base_path / 'processed' / 'dataset_medidores.parquet'
df = pd.read_parquet(input_path)

if output_path.exists():
    output_path.unlink()
    print("Archivo anterior eliminado")

print('Shape original:', df.shape)

Archivo anterior eliminado
Shape original: (1434844, 15)


In [51]:
df['FirstIntervalDateTime'] = pd.to_datetime(
    df['FirstIntervalDateTime'],
    format='%m%d%Y%I%M%S%p'
)

In [52]:
schema = pa.schema([
    ("Meter ID", pa.string()),
    ("Datetime", pa.timestamp("us")),
    ("Units", pa.string()),
    ("Value", pa.float64()),
    ("Quality", pa.string())
])

chunk_size = 20000
writer = None

for i in range(0, len(df), chunk_size):
    print(f"Procesando chunk {i} - {i+chunk_size}")
    
    df_chunk = df.iloc[i:i+chunk_size].copy()
    
    df_chunk["Data_list"] = df_chunk["Data"].str.split("~")
    df_chunk = df_chunk.explode("Data_list")
    df_chunk = df_chunk[df_chunk["Data_list"] != ""]
    
    df_chunk["Flag"] = df_chunk["Data_list"].str.extract(r'^([A-Z])')
    df_chunk["Value"] = df_chunk["Data_list"].str.extract(r'([0-9.]+)')
    
    df_chunk["Value"] = pd.to_numeric(df_chunk["Value"], errors="coerce")
    
    df_chunk.loc[df_chunk["Flag"] == "N", "Value"] = np.nan
    
    df_chunk["interval_index"] = df_chunk.groupby(
        ["Meter ID", "Units", "FirstIntervalDateTime"]
    ).cumcount()
    
    df_chunk["Datetime"] = (
        df_chunk["FirstIntervalDateTime"] +
        pd.to_timedelta(df_chunk["interval_index"] * 15, unit="m")
    )
    
    df_chunk["Quality"] = df_chunk["Flag"].fillna("").replace({
        "": "valid",
        "E": "estimated",
        "A": "adjusted",
        "R": "raw",
        "N": "missing"
    })
    
    df_chunk = df_chunk[[
        "Meter ID",
        "Datetime",
        "Units",
        "Value",
        "Quality"
    ]]
    
    df_chunk = df_chunk.reset_index(drop=True)

    table = pa.Table.from_pandas(
        df_chunk,
        schema=schema,
        preserve_index=False
    )   
    
    if writer is None:
        writer = pq.ParquetWriter(output_path, schema)
    
    writer.write_table(table)

if writer:
    writer.close()

print("✅ Dataset completo con estructura conservada")

Procesando chunk 0 - 20000
Procesando chunk 20000 - 40000
Procesando chunk 40000 - 60000
Procesando chunk 60000 - 80000
Procesando chunk 80000 - 100000
Procesando chunk 100000 - 120000
Procesando chunk 120000 - 140000
Procesando chunk 140000 - 160000
Procesando chunk 160000 - 180000
Procesando chunk 180000 - 200000
Procesando chunk 200000 - 220000
Procesando chunk 220000 - 240000
Procesando chunk 240000 - 260000
Procesando chunk 260000 - 280000
Procesando chunk 280000 - 300000
Procesando chunk 300000 - 320000
Procesando chunk 320000 - 340000
Procesando chunk 340000 - 360000
Procesando chunk 360000 - 380000
Procesando chunk 380000 - 400000
Procesando chunk 400000 - 420000
Procesando chunk 420000 - 440000
Procesando chunk 440000 - 460000
Procesando chunk 460000 - 480000
Procesando chunk 480000 - 500000
Procesando chunk 500000 - 520000
Procesando chunk 520000 - 540000
Procesando chunk 540000 - 560000
Procesando chunk 560000 - 580000
Procesando chunk 580000 - 600000
Procesando chunk 600000